# Day 12 — Feature Engineering & Feature Selection

## Task 1 — من الأعمدة إلى Features

### 1. الفرق بين Feature Engineering وFeature Selection

Feature Engineering يعني إنشاء خصائص جديدة من البيانات الموجودة، أما Feature Selection فهو اختيار الخصائص المفيدة واستبعاد غير المناسب أو المتكرر منها.

مثالي من مجال التعليم: التنبؤ بدرجة الطالبة في الاختبار النهائي.

- Target: درجة الاختبار النهائي.
- الخصائص الموجودة: ساعات المذاكرة أسبوعيًا، وعدد الغيابات، وعدد الواجبات المكتملة، ودرجة الاختبار السابق.
- الخصائص الجديدة التي يمكن توليدها: نسبة الحضور، ونسبة إنجاز الواجبات، ومتوسط ساعات المذاكرة يوميًا.

أستخدم المعلومات المتاحة قبل الاختبار، وأستبعد اسم الطالبة ورقمها لأنهما للتعريف بها ولا يعبران عن مستواها الدراسي.

### 2. طرق اختيار الخصائص

| الطريقة | الفكرة باختصار | مثال | ميزة | عيب |
|---|---|---|---|---|
| Filter | أقيم الخصائص بإحصاءات دون تدريب موديل | الارتباط أو اختبار كاي تربيع | سريعة وسهلة التفسير | قد لا تكشف تأثير الخصائص مع بعضها |
| Wrapper | أجرب مجموعات من الخصائص وأقارن أداء الموديل | RFE | تختار خصائص مناسبة للموديل المستخدم | تحتاج وقتًا وحسابات أكثر |
| Embedded | يتم اختيار الخصائص أثناء تدريب الموديل | Lasso | تجمع التدريب والاختيار في خطوة واحدة | تتأثر بنوع الموديل وإعداداته |

نستخدم Filter هذا الأسبوع حتى نفهم علاقة الأعمدة بالهدف، ونبني قرارات الاختيار على أرقام واضحة قبل تدريب الموديل.

### 3. تسرّب البيانات Data Leakage

أحدد صلاحية كل عمود حسب توفره وقت تقديم طلب القرض:

| العمود | قراري | السبب |
|---|---|---|
| Disbursement_Date | أستبعده | تاريخ صرف القرض يأتي بعد الموافقة، ولا يكون معروفًا عند تقديم الطلب |
| Customer_Since_Years | أستخدمه | مدة تعامل العميل مع البنك معلومة متاحة، بشرط حسابها حتى تاريخ تقديم الطلب |
| Manager_Final_Decision | أستبعده | قرار المدير النهائي يأتي بعد دراسة الطلب، وقد يكشف النتيجة التي أحاول التنبؤ بها |

استخدام معلومات لا تتوفر إلا بعد القرار يسبب تسرّب البيانات. قد تظهر نتائج التقييم مرتفعة، لكن عند تطبيق الموديل على طلبات جديدة ينخفض أداؤه لأنه اعتمد على معلومات غير متاحة وقت التنبؤ.

### 4. هل الارتباط القريب من الصفر يعني أن العمود غير مفيد؟

لا، لأن ارتباط بيرسون يقيس العلاقة الخطية، وقد تكون العلاقة غير خطية والعمود ما زال مفيدًا.

مثال من حياتي: كمية الماء التي أسقي بها النبتة. الماء القليل يضعف نموها، والكمية المناسبة تساعدها، لكن الماء الزائد يضرها أيضًا.

لذلك العلاقة ليست خطًا مستقيمًا؛ زيادة الماء لا تعني دائمًا زيادة النمو. قد يكون الارتباط الخطي قريبًا من الصفر، رغم أن كمية الماء تؤثر في النبتة. لهذا لا أحذف العمود اعتمادًا على الارتباط وحده.

## Task 2 — إنشاء خصائص تعبّر عن القدرة على السداد

سأجمع دخل المتقدم والشريك، ثم أحسب نسبة مبلغ القرض والقسط الشهري إلى الدخل الكلي، حتى أدرس القدرة على السداد.

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

Saving loan_clean.csv to loan_clean.csv


In [2]:
df = pd.read_csv("loan_clean.csv")

print("حجم البيانات:", df.shape)
print("أسماء الأعمدة:")
print(df.columns.tolist())

display(df.head())

حجم البيانات: (614, 13)
أسماء الأعمدة:
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1,Urban,1
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1,Rural,0
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1,Urban,1
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1,Urban,1
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1,Urban,1


In [3]:
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)

# تحويل مبلغ القرض من الآلاف إلى الريال
loan_amount_riyals = df["LoanAmount"] * 1000

# تجنب القسمة على صفر
income = df["Total_Income"].replace(0, np.nan)
term = df["Loan_Amount_Term"].replace(0, np.nan)

df["Loan_to_Income"] = loan_amount_riyals / income

# قسط شهري تقريبي بدون فوائد
df["Monthly_Payment"] = loan_amount_riyals / term

df["Payment_to_Income"] = df["Monthly_Payment"] / income

new_features = [
    "Total_Income",
    "Has_Coapplicant",
    "Loan_to_Income",
    "Monthly_Payment",
    "Payment_to_Income"
]

display(df[new_features].head())

display(df[["Loan_to_Income", "Payment_to_Income"]].describe())

,Total_Income,Has_Coapplicant,Loan_to_Income,Monthly_Payment,Payment_to_Income
0,5849.0,0,21.884083,355.555556,0.060789
1,6091.0,1,21.014612,355.555556,0.058374
2,3000.0,0,22.000000,183.333333,0.061111
3,4941.0,1,24.286582,333.333333,0.067463
4,6000.0,0,23.500000,391.666667,0.065278


,Loan_to_Income,Payment_to_Income
count,614.000000,614.000000
mean,23.876787,0.077391
std,8.707161,0.079017
min,2.522916,0.007008
25%,19.220552,0.055302
50%,24.129209,0.068970
75%,28.158020,0.081695
max,82.711645,1.236873


In [4]:
# تحويل الموافقة إلى رقم لحساب النسبة
print("قيم قرار القرض:", df["Loan_Status"].unique())

df["Approved"] = df["Loan_Status"].map({"Y": 1, "N": 0})

df["Income_Group"] = pd.qcut(
    df["ApplicantIncome"], 4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

df["Payment_Group"] = pd.qcut(
    df["Payment_to_Income"], 4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

for column in ["Income_Group", "Payment_Group", "Has_Coapplicant"]:
    print(f"\nنسبة الموافقة حسب {column}:")
    summary = df.groupby(column, observed=True)["Approved"].agg(
        ["count", "mean"]
    )
    summary["Approval_Rate_%"] = summary["mean"] * 100
    display(summary[["count", "Approval_Rate_%"]].round(2))

قيم قرار القرض: [1 0]

نسبة الموافقة حسب Income_Group:


,count,Approval_Rate_%
Income_Group,,
Q1,0,NaN
Q2,0,NaN
Q3,0,NaN
Q4,0,NaN



نسبة الموافقة حسب Payment_Group:


,count,Approval_Rate_%
Payment_Group,,
Q1,0,NaN
Q2,0,NaN
Q3,0,NaN
Q4,0,NaN



نسبة الموافقة حسب Has_Coapplicant:


,count,Approval_Rate_%
Has_Coapplicant,,
0,0,NaN
1,0,NaN


In [7]:
# تحويل الموافقة إلى رقم لحساب النسبة
print("قيم قرار القرض:", df["Loan_Status"].unique())

df["Approved"] = df["Loan_Status"]

df["Income_Group"] = pd.qcut(
    df["ApplicantIncome"], 4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

df["Payment_Group"] = pd.qcut(
    df["Payment_to_Income"], 4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

for column in ["Income_Group", "Payment_Group", "Has_Coapplicant"]:
    print(f"\nنسبة الموافقة حسب {column}:")
    summary = df.groupby(column, observed=True)["Approved"].agg(
        ["count", "mean"]
    )
    summary["Approval_Rate_%"] = summary["mean"] * 100
    display(summary[["count", "Approval_Rate_%"]].round(2))

قيم قرار القرض: [1 0]

نسبة الموافقة حسب Income_Group:


,count,Approval_Rate_%
Income_Group,,
Q1,154,68.83
Q2,153,68.63
Q3,153,69.28
Q4,154,68.18



نسبة الموافقة حسب Payment_Group:


,count,Approval_Rate_%
Payment_Group,,
Q1,154,64.29
Q2,153,71.24
Q3,153,73.86
Q4,154,65.58



نسبة الموافقة حسب Has_Coapplicant:


,count,Approval_Rate_%
Has_Coapplicant,,
0,273,64.84
1,341,71.85


In [6]:
print(df["Loan_Status"].value_counts(dropna=False))

Loan_Status
1    422
0    192
Name: count, dtype: int64


### ملاحظاتي على النتائج

متوسط نسبة القسط إلى الدخل حوالي 7.74%، وأعلى نسبة وصلت إلى 123.69%. هذه الحالة تحتاج مراجعة مبلغ القرض ومدة السداد والدخل، لأن القسط التقريبي تجاوز الدخل الشهري، ولا أحذفها قبل التأكد من سببها.

اختلفت نسبة الموافقة بين فئات Payment_to_Income: كانت 64.29% في Q1، و71.24% في Q2، و73.86% في Q3، ثم انخفضت إلى 65.58% في Q4. الفرق بين أعلى وأقل فئة حوالي 9.57 نقطة مئوية، لكن العلاقة لم تكن باتجاه واحد.

كانت نسبة الموافقة 71.85% لمن لديهم دخل شريك، مقابل 64.84% لمن ليس لديهم دخل شريك، بفارق 7.01 نقاط مئوية.

توقعت أن تنخفض الموافقة كلما ارتفعت نسبة القسط إلى الدخل، لكن النتيجة لم تتبع هذا النمط بانتظام. وجود دخل شريك ارتبط بنسبة موافقة أعلى في هذه العينة، لكنه وحده لا يثبت سبب الموافقة.

In [8]:
income_summary = df.groupby(
    "Income_Group", observed=True
)["Approved"].agg(["count", "mean"])

income_summary["Approval_Rate_%"] = income_summary["mean"] * 100

display(income_summary[["count", "Approval_Rate_%"]].round(2))

,count,Approval_Rate_%
Income_Group,,
Q1,154,68.83
Q2,153,68.63
Q3,153,69.28
Q4,154,68.18


### مقارنة الدخل بالخصائص الجديدة

تراوحت نسب الموافقة بين فئات دخل المتقدم من 68.18% إلى 69.28%، بفارق حوالي 1.10 نقطة مئوية فقط؛ لذلك لم يظهر اختلاف كبير بين فئات الدخل.

أما Payment_to_Income فأظهر فرقًا قدره 9.57 نقطة مئوية بين أعلى وأقل فئة، وHas_Coapplicant أظهر فرقًا قدره 7.01 نقاط مئوية.

الخصائص الجديدة أظهرت اختلافات أوضح من دخل المتقدم وحده في هذه العينة، لكن ذلك لا يكفي لإثبات أنها ستحسن أداء الموديل.

In [9]:
loan_features = df.drop(columns=["Approved"]).copy()

loan_features.to_csv("loan_features.csv", index=False)

print("تم حفظ الملف، حجمه:", loan_features.shape)
files.download("loan_features.csv")

تم حفظ الملف، حجمه: (614, 20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 3 — اختيار الخصائص باستخدام Filter

سأفحص عدد القيم المختلفة في كل عمود، ثم أقارن نسب الموافقة والارتباطات، حتى أحدد الأعمدة المناسبة وأوثق سبب الاحتفاظ بها أو استبعادها.

In [10]:
unique_counts = loan_features.nunique(dropna=False)

display(unique_counts.to_frame("عدد القيم المختلفة"))

print("الأعمدة التي تختلف قيمتها في كل صف:")
print(unique_counts[unique_counts == len(loan_features)].index.tolist())

print("\nالأعمدة التي تحتوي على قيمة واحدة:")
print(unique_counts[unique_counts == 1].index.tolist())

,عدد القيم المختلفة
Loan_ID,614
Gender,2
Married,2
Dependents,4
Education,2
Self_Employed,2
ApplicantIncome,505
CoapplicantIncome,287
LoanAmount,203
Loan_Amount_Term,10


الأعمدة التي تختلف قيمتها في كل صف:
['Loan_ID']

الأعمدة التي تحتوي على قيمة واحدة:
[]


In [11]:
categorical_columns = [
    "Gender", "Married", "Dependents",
    "Education", "Self_Employed", "Credit_History",
    "Property_Area", "Has_Coapplicant",
    "Income_Group", "Payment_Group"
]

results = []

for column in categorical_columns:
    rates = loan_features.groupby(
        column, observed=True, dropna=False
    )["Loan_Status"].agg(["size", "mean"])

    rates["Approval_Rate_%"] = rates["mean"] * 100

    print(f"\n{column}")
    display(rates[["size", "Approval_Rate_%"]].round(2))

    results.append({
        "Feature": column,
        "Gap_percentage_points":
            rates["Approval_Rate_%"].max()
            - rates["Approval_Rate_%"].min()
    })

gap_table = pd.DataFrame(results).sort_values(
    "Gap_percentage_points", ascending=False
)

print("\nترتيب الأعمدة حسب فرق نسب الموافقة:")
display(gap_table.round(2))


Gender


,size,Approval_Rate_%
Gender,,
Female,112,66.96
Male,502,69.12



Married


,size,Approval_Rate_%
Married,,
No,213,62.91
Yes,401,71.82



Dependents


,size,Approval_Rate_%
Dependents,,
0,360,68.61
1,102,64.71
2,101,75.25
3,51,64.71


... (12 outputs left out)


Payment_Group


,size,Approval_Rate_%
Payment_Group,,
Q1,154,64.29
Q2,153,71.24
Q3,153,73.86
Q4,154,65.58



ترتيب الأعمدة حسب فرق نسب الموافقة:


,Feature,Gap_percentage_points
5,Credit_History,71.71
6,Property_Area,15.37
2,Dependents,10.54
3,Education,9.64
9,Payment_Group,9.57
1,Married,8.91
7,Has_Coapplicant,7.01
0,Gender,2.16
8,Income_Group,1.10
4,Self_Employed,0.50


In [12]:
numeric_data = loan_features.select_dtypes(include="number")

target_corr = numeric_data.corr()["Loan_Status"].drop("Loan_Status")

target_corr = target_corr.reindex(
    target_corr.abs().sort_values(ascending=False).index
)

print("ارتباط الأعمدة الرقمية بالهدف:")
display(target_corr.to_frame("Correlation").round(4))

income_corr = loan_features[
    ["ApplicantIncome", "Total_Income"]
].corr().iloc[0, 1]

print("\nالارتباط بين دخل المتقدم والدخل الكلي:")
print(round(income_corr, 4))

ارتباط الأعمدة الرقمية بالهدف:


,Correlation
Credit_History,0.2802
Loan_to_Income,-0.0861
Has_Coapplicant,0.0752
CoapplicantIncome,-0.0592
Payment_to_Income,-0.0514
LoanAmount,-0.0332
Total_Income,-0.0313
Loan_Amount_Term,-0.0225
Monthly_Payment,-0.0118
Dependents,0.0101



الارتباط بين دخل المتقدم والدخل الكلي:
0.893


In [13]:
display(
    gap_table[
        gap_table["Feature"].isin(["Credit_History", "Property_Area"])
    ].round(2)
)

,Feature,Gap_percentage_points
5,Credit_History,71.71
6,Property_Area,15.37


### ملاحظاتي على الارتباط

أعلى ارتباط رقمي بالهدف كان للسجل الائتماني Credit_History بقيمة 0.2802.

ارتباط دخل المتقدم بالهدف ضعيف جدًا ويساوي -0.0047، كما أن نسب الموافقة بين فئات دخله متقاربة بفارق 1.10 نقطة مئوية.

الارتباط بين ApplicantIncome وTotal_Income بلغ 0.893، لذلك أختار الاحتفاظ بالدخل الكلي واستبعاد دخل المتقدم لتقليل التشابه في هذه النسخة الأولية؛ فالدخل الكلي يشمل دخل المتقدم والشريك.

لا أستبعد بقية الأعمدة لمجرد ضعف ارتباطها الخطي؛ أقارن أيضًا نسب الموافقة بين الفئات ومعنى كل خاصية.

### جدول قرارات اختيار الخصائص

هذه قرارات أولية مبنية على الإحصاءات ومعنى الأعمدة، ولم أختبر أثرها على أداء موديل بعد.

| العمود | القرار | السبب والدليل |
|---|---|---|
| Loan_ID | أستبعده | معرّف مختلف لكل طلب من الطلبات الـ614 |
| Gender | أستبعده من المدخلات | فرق الموافقة 2.16 نقطة فقط، وأحتفظ به منفصلًا لمراجعة العدالة |
| Married | أحتفظ به مبدئيًا | فرق الموافقة 8.91 نقاط، مع ضرورة مراجعة ملاءمة استخدامه |
| Dependents | أحتفظ به | فرق الموافقة بين فئاته 10.54 نقاط رغم ضعف الارتباط الخطي 0.0101 |
| Education | أحتفظ به مبدئيًا | فرق الموافقة 9.64 نقاط |
| Self_Employed | أستبعده مبدئيًا | فرق الموافقة 0.50 نقطة فقط |
| ApplicantIncome | أستبعده | ارتباطه بالدخل الكلي 0.893، وفجوة فئات دخله 1.10 نقطة فقط |
| CoapplicantIncome | أحتفظ به مبدئيًا | يوضح مقدار مساهمة الشريك؛ ارتباطه بالهدف -0.0592 |
| LoanAmount | أحتفظ به | يمثل مبلغ الالتزام؛ ارتباطه بالهدف -0.0332 لا يكفي وحده لحذفه |
| Loan_Amount_Term | أحتفظ به | يحدد مدة السداد ويؤثر في حساب القسط؛ ارتباطه -0.0225 |
| Credit_History | أحتفظ به | ارتباطه 0.2802 وفجوة الموافقة 71.71 نقطة |
| Property_Area | أحتفظ به مبدئيًا | فرق الموافقة 15.37 نقطة، مع مراجعة احتمال ارتباطه بعوامل اجتماعية |
| Total_Income | أحتفظ به | يجمع دخل المتقدم والشريك ويستخدم في حساب نسب الالتزام؛ ارتباطه بالهدف -0.0313 |
| Has_Coapplicant | أحتفظ به | فرق الموافقة 7.01 نقاط وارتباطه 0.0752 |
| Loan_to_Income | أحتفظ به | يعبر عن حجم القرض مقارنة بالدخل؛ ارتباطه -0.0861 |
| Monthly_Payment | أحتفظ به مبدئيًا | يوضح القسط التقريبي؛ ارتباطه -0.0118، وأراجع تكراره مع الخصائص المشتقة لاحقًا |
| Payment_to_Income | أحتفظ به | فجوة فئاته 9.57 نقاط رغم أن ارتباطه الخطي -0.0514 |
| Income_Group | أستبعده من المدخلات | أنشأته للمقارنة، وفجوة الموافقة بين فئاته 1.10 نقطة |
| Payment_Group | أستبعده من المدخلات | أستخدمه للتحليل وأحتفظ بالنسبة الرقمية الأصلية |
| Loan_Status | أحتفظ به كهدف فقط | هو النتيجة المطلوب التنبؤ بها، ولا يدخل ضمن الخصائص |

### رأيي في استخدام Gender

أستبعد Gender من مدخلات الموديل، وأحتفظ به منفصلًا لمقارنة النتائج بين المجموعات. حذف العمود وحده لا يضمن العدالة، فقد تنقل أعمدة أخرى فروقًا مشابهة. فرق الموافقة 2.16 نقطة وصف للعينة، ولا يثبت وحده وجود تحيز أو غيابه.

In [14]:
# الاحتفاظ ببيانات مراجعة العدالة منفصلة عن مدخلات الموديل
fairness_audit = loan_features[
    ["Loan_ID", "Gender", "Loan_Status"]
].copy()

excluded_columns = [
    "Loan_ID",
    "Gender",
    "Self_Employed",
    "ApplicantIncome",
    "Income_Group",
    "Payment_Group"
]

model_ready = loan_features.drop(columns=excluded_columns).copy()

model_ready.to_csv("model_ready.csv", index=False)

print("حجم البيانات المختارة:", model_ready.shape)
print("الأعمدة النهائية:")
print(model_ready.columns.tolist())

files.download("model_ready.csv")

حجم البيانات المختارة: (614, 14)
الأعمدة النهائية:
['Married', 'Dependents', 'Education', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status', 'Total_Income', 'Has_Coapplicant', 'Loan_to_Income', 'Monthly_Payment', 'Payment_to_Income']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Task 5 — حساسات مسبار الحياة

سأقارن الحساسات باستخدام الارتباط ومتوسط القراءات، ثم أختار ثلاثة حساسات وأوضح سبب استبعاد البقية.

In [15]:
rng = np.random.default_rng(2026)
n = 300
life = rng.integers(0, 2, n)

probes = pd.DataFrame({
    "probe_id": [f"P{i:03d}" for i in range(n)],
    "mission_code": "AX-7",
    "methane_ppm": np.round(
        5 + life * 12 + rng.normal(0, 3, n), 1
    ),
    "temperature_c": np.round(
        -40 + life * 25 + rng.normal(0, 15, n), 1
    ),
    "radiation": np.round(rng.normal(100, 10, n), 1),
    "humidity": np.round(
        20 + life * 6 + rng.normal(0, 8, n), 1
    ),
    "life_detected": life
})

probes["temperature_f"] = probes["temperature_c"] * 9 / 5 + 32

display(probes.head())
print("حجم البيانات:", probes.shape)
display(probes.nunique().to_frame("Unique_values"))

,probe_id,mission_code,methane_ppm,temperature_c,radiation,humidity,life_detected,temperature_f
0,P000,AX-7,17.7,-18.0,71.4,19.2,1,-0.40
1,P001,AX-7,2.2,-61.9,108.9,27.9,0,-79.42
2,P002,AX-7,9.7,-37.4,102.3,24.8,0,-35.32
3,P003,AX-7,21.6,-24.5,119.1,20.4,1,-12.10
4,P004,AX-7,6.7,-54.5,101.9,14.7,0,-66.10


حجم البيانات: (300, 8)


,Unique_values
probe_id,300
mission_code,1
methane_ppm,160
temperature_c,246
radiation,212
humidity,192
life_detected,2
temperature_f,246


In [16]:
sensors = [
    "methane_ppm", "temperature_c",
    "temperature_f", "radiation", "humidity"
]

sensor_corr = probes[sensors].corrwith(probes["life_detected"])
sensor_corr = sensor_corr.reindex(
    sensor_corr.abs().sort_values(ascending=False).index
)

print("ترتيب ارتباط الحساسات بوجود الحياة:")
display(sensor_corr.to_frame("Correlation").round(4))

print("\nالارتباط بين درجتي الحرارة:")
print(probes["temperature_c"].corr(probes["temperature_f"]).round(4))

print("\nمتوسط قراءات الحساسات حسب وجود الحياة:")
display(
    probes.groupby("life_detected")[sensors].mean().round(2)
)

ترتيب ارتباط الحساسات بوجود الحياة:


,Correlation
methane_ppm,0.8820
temperature_f,0.6430
temperature_c,0.6430
humidity,0.3975
radiation,0.1098



الارتباط بين درجتي الحرارة:
1.0

متوسط قراءات الحساسات حسب وجود الحياة:


,methane_ppm,temperature_c,temperature_f,radiation,humidity
life_detected,,,,,
0,5.38,-40.31,-40.55,98.74,19.14
1,17.04,-14.78,5.39,100.96,25.63


### قراري لاختيار الحساسات

أستبعد probe_id لأنه معرّف فريد له 300 قيمة مختلفة، وأستبعد mission_code لأنه ثابت بقيمة واحدة.

أختار الحساسات الثلاثة التالية بالترتيب:

1. methane_ppm: متوسطه 17.04 عند وجود الحياة، مقابل 5.38 عند عدم وجودها، ويأتي أولًا في ترتيب الارتباط بالهدف.
2. temperature_c: متوسطه -14.78 عند وجود الحياة، مقابل -40.31 عند عدم وجودها، ويأتي بعد الميثان في ترتيب الارتباط.
3. humidity: متوسطه 25.63 عند وجود الحياة، مقابل 19.14 عند عدم وجودها، ويضيف قياسًا مختلفًا عن الحرارة والميثان.

أستبعد temperature_f لأن ارتباطه مع temperature_c يساوي 1.0، فهو نفس قياس الحرارة بوحدة مختلفة، ولا نحتاج إلى الاثنين.

أستبعد radiation لأن ارتباطه بالهدف ضعيف ويساوي 0.1098، ومتوسطه متقارب نسبيًا بين المجموعتين: 100.96 مع وجود الحياة و98.74 دونها.

### كيف أتحقق من الاختيار لاحقًا؟

أجرب Wrapper باستخدام اختيار أمامي للخصائص مع تحقق متقاطع، وأحدد العدد بثلاثة حساسات. أقارن أداء المجموعات على بيانات التدريب، ثم أقيّم الاختيار النهائي على بيانات اختبار منفصلة، حتى أتأكد أن الاختيار لا يعتمد على هذه العينة وحدها.

In [17]:
selected_sensors = ["methane_ppm", "temperature_c", "humidity"]

print("الحساسات الثلاثة المختارة وارتباطها بالهدف:")
display(
    sensor_corr.loc[selected_sensors]
    .to_frame("Correlation")
    .round(4)
)

الحساسات الثلاثة المختارة وارتباطها بالهدف:


,Correlation
methane_ppm,0.8820
temperature_c,0.6430
humidity,0.3975
